1. 明确入库目标
   ↓
2. 读取所有 *_chunks.jsonl
   ↓
3. 每一行 JSON 转成 Document
   ↓
4. 生成稳定 ID
   ↓
5. 创建 / 重置 Chroma collection
   ↓
6. 批量 add_documents 入库
   ↓
7. 验证 collection 数量
   ↓
8. 用 similarity / MMR / metadata filter 检索验证


In [2]:
from pathlib import Path
""" 
创建一个 可以被python理解并操作的路径对象 注意r"rag_all\ingest_Chroma.ipynb" 或者 /
"""
data_dir = Path("rag_all/ingest_Chroma.ipynb")



In [3]:
# 获取当前工作目录
current_dir = Path.cwd()
print(current_dir)
print(current_dir.exists()) # Path对象可以通过exist查看是否存在

e:\2026\05\all_api\rag_all
True


In [4]:
# 这是.py文件所在的目录
file_dir = Path(__file__).parent
print(file_dir)

NameError: name '__file__' is not defined

In [5]:
""" 
地址拼接
传统方法是os.path.join
推荐做法是 / 
"""
import os
path = os.path.join(current_dir, "example.txt")
print(path)
path = current_dir / "example.txt"
print(path)

e:\2026\05\all_api\rag_all\example.txt
e:\2026\05\all_api\rag_all\example.txt


In [6]:
# 获取文件名、后缀、父目录
path = Path("rag_all/ingest_Chroma.ipynb")
print(path.name) 
print(path.suffix) 
print(path.stem)

ingest_Chroma.ipynb
.ipynb
ingest_Chroma


In [7]:
# 创建文件夹
data_dir = Path("data/raw/text")
data_dir.mkdir(parents=True, exist_ok=True)

In [8]:
# 遍历文件夹
data_dir = Path.cwd()  
for file in data_dir.iterdir():
    print(file)

e:\2026\05\all_api\rag_all\chroma
e:\2026\05\all_api\rag_all\chroma_db
e:\2026\05\all_api\rag_all\chroma_db_from_documents
e:\2026\05\all_api\rag_all\chroma_db_from_texts
e:\2026\05\all_api\rag_all\data
e:\2026\05\all_api\rag_all\Document.png
e:\2026\05\all_api\rag_all\generated_question_bank
e:\2026\05\all_api\rag_all\ingest_Chroma.ipynb
e:\2026\05\all_api\rag_all\ingest_question_bank.py
e:\2026\05\all_api\rag_all\mmr_search.png
e:\2026\05\all_api\rag_all\rag_flow.ipynb
e:\2026\05\all_api\rag_all\rag_to_store.ipynb
e:\2026\05\all_api\rag_all\requirements
e:\2026\05\all_api\rag_all\vectorstore.py


In [9]:
# 只会查找当前目录
for file in data_dir.glob("*.ipynb"):
    print(file)

e:\2026\05\all_api\rag_all\ingest_Chroma.ipynb
e:\2026\05\all_api\rag_all\rag_flow.ipynb
e:\2026\05\all_api\rag_all\rag_to_store.ipynb


In [10]:
# 会递归查找所有子目录
for file in data_dir.rglob("*.ipynb"):
    print(file)

e:\2026\05\all_api\rag_all\ingest_Chroma.ipynb
e:\2026\05\all_api\rag_all\rag_flow.ipynb
e:\2026\05\all_api\rag_all\rag_to_store.ipynb


![1](Document.png)

In [11]:
from langchain_core.documents import Document
""" 
把一行Json转换成Document
这里的 obj 是一个 字典对象 dict，表示从 JSON / JSONL 文件中读取出来的一条数据。
这个函数接收一个字典 obj，再接收一个来源文件名 source_file，最后返回一个 LangChain 的 Document 对象。

复习一下 dict
dict = {
    "key1": "value1",
    "key2": "value2"}
获取数据的方法是
value1 = dict.get("key1", "default_value")
"""
def json_to_document(obj:dict , source_file: str) -> Document:
    role = obj.get("role", "")
    topic = obj.get("topic", "")
    chunk_type = obj.get("chunk_type", "") 
    content = obj.get("content", "")
    key_points = obj.get("key_points", [])
    related_topics = obj.get("related_topics", [])
    tags = obj.get("tags", [])

    page_content = (
        f"[知识点]: {topic}\n"
        f"[知识分类]: {role}\n"
        f"[内容]: {content}\n"
        f"[关键点]: {', '.join(key_points)}\n"
        f"[相关主题]: {', '.join(related_topics)}\n"
        f"[标签]: {', '.join(tags)}\n"
    )

    metadata = {
        "source": source_file,
        "role": role,
        "topic": topic,
        "chunk_type": chunk_type,   
    }

    return Document(page_content=page_content, metadata=metadata)

In [12]:
import json
def load_documents_and_ids_from_jsonl(file_path: Path):
    """ 
    打开文件地址，读取文件内容，并把每一行的 JSON 字符串转换成 Document 对象，最后返回一个 Document 对象的列表。
    处理单个jsonl文件
    返回Document 和 ids
    """
    docs = []
    ids = []
    """ 
    打开文件，读取文件内容，并在用完之后自动关闭文件 可以是Path,也可以是str
    """
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip() # 去掉字符串首尾的空白字符
            if not line:
                continue    
            obj = json.loads(line)
            """
            把一行 JSON 字符串(str)转换成 Python 字典
            load str 的意思
            是json库中的函数,用于将 JSON 格式的字符串解析成 Python 对象（通常是字典或列表）。它接受一个字符串参数，并返回解析后的 Python 对象。
            
            loads 处理的是str
            load 处理的是文件对象
            """
            doc = json_to_document(obj, source_file=file_path.name)
            docs.append(doc)
            doc_id = f"{file_path.stem}_{len(docs)}"
            ids.append(doc_id)
    return docs, ids

In [13]:
import json
from pathlib import Path
import os

from langchain_core.documents import Document

data_dir = Path.cwd()
print(type(data_dir))
data_dir = os.path.join(data_dir, "generated_question_bank")
print(type(data_dir))
data_dir = Path(data_dir)
print(type(data_dir))
print(data_dir)

""" 
generator迭代器只能被遍历一次,len(list(generator))会将generator转换成list,
之后generator就被消耗掉了,所以无法再次遍历。
所以解决方法是提前转换成list类对象
"""
# chunk_files = data_dir.glob("*_chunks.jsonl")
# print(type(chunk_files)) # 这是一个generator迭代器
# print(f"nums = {len(list(chunk_files))}")
# for chunk in chunk_files:
#     print(chunk.name)

chunk_files = list(data_dir.glob("*_chunks.jsonl"))
print(type(chunk_files)) # 这是一个generator迭代器
print(f"nums = {len(chunk_files)}")

all_docs = []
all_ids = []

for chunk in chunk_files:
    docs, ids = load_documents_and_ids_from_jsonl(chunk)
    print(f"load {chunk.name} successfully!")
    print(type(docs))
    # print(all_ids[0])
    all_docs.extend(docs)
    all_ids.extend(ids)


<class 'pathlib.WindowsPath'>
<class 'str'>
<class 'pathlib.WindowsPath'>
e:\2026\05\all_api\rag_all\generated_question_bank
<class 'list'>
nums = 9
load cpp_chunks.jsonl successfully!
<class 'list'>
load cs_fundamentals_chunks.jsonl successfully!
<class 'list'>
load embedded_chunks.jsonl successfully!
<class 'list'>
load frontend_chunks.jsonl successfully!
<class 'list'>
load go_chunks.jsonl successfully!
<class 'list'>
load java_chunks.jsonl successfully!
<class 'list'>
load llm_core_tech_chunks.jsonl successfully!
<class 'list'>
load python_backend_chunks.jsonl successfully!
<class 'list'>
load python_chunks.jsonl successfully!
<class 'list'>


In [14]:
print(f"total docs: {len(all_docs)}")
print(type(all_docs[0]))
print(all_docs[0].metadata)
print(all_ids[0])

total docs: 1374
<class 'langchain_core.documents.base.Document'>
{'source': 'cpp_chunks.jsonl', 'role': 'cpp', 'topic': 'C++ Syntax', 'chunk_type': 'principle'}
cpp_chunks_1


In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma

emb = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
persist_dir = "chroma"
collection_name = "knowledge_chunk"
db = Chroma(
    persist_directory=persist_dir,
    embedding_function=emb,
    collection_name=collection_name
)

e:\miniconda\envs\langchain2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4983.02it/s]


In [16]:
db.add_documents(all_docs, ids=all_ids)

print("入库完成")
print("当前 collection 数量:", db._collection.count())

入库完成
当前 collection 数量: 1374


In [17]:
result = db._collection.get(
    ids = ["cpp_chunks_11"]
)
for doc_id , doc_text , mete in zip(result["ids"] , result["documents"],result["metadatas"]):
    print(f"ID: {doc_id}\npage_content:\n{doc_text}\nmetedata: {mete}\n")
    

ID: cpp_chunks_11
page_content:
[知识点]: C++20
[知识分类]: cpp
[内容]: C++20 引入的模块（Modules）是 C++ 语言历史上最重要的编译模型革新之一，其核心原理是彻底取代传统的头文件包含（#include）机制，通过显式声明的接口（export）和独立的编译单元来组织代码。模块在编译时被解析为独立的抽象语法树（AST），避免了头文件带来的预处理开销、宏污染、重复编译和命名空间污染等问题。模块的编译过程分为两个阶段：首先编译模块接口单元（.cppm 或 .ixx）生成模块元数据（.pcm），然后在编译其他源文件时直接导入这些元数据，无需重新解析源码。这显著提升了编译速度，尤其适用于大型项目。模块的原理依赖于 C++20 的新关键字（如 export、import、module）和编译器对模块元数据的管理，其设计目标是实现更安全的代码隔离和更快的增量构建。常见误区包括误以为模块可以完全替代头文件（实际需逐步迁移），或忽略模块接口的显式导出规则导致链接错误。工程实践中，模块适用于库开发和大型项目，但需注意工具链支持（如 MSVC、GCC 11+、Clang 10+）和构建系统（如 CMake 3.28+）的适配。
[关键点]: 模块通过 export 和 import 关键字显式定义接口，避免头文件的宏和重复编译问题。, 编译过程分为模块接口编译生成 .pcm 元数据和后续导入，提升增量构建效率。, 模块提供更好的代码隔离和安全性，减少命名冲突和预处理开销。, 工程迁移需注意工具链兼容性和构建系统配置，避免混合使用头文件和模块。, 模块不适用于所有场景，如模板特化或跨平台库可能仍需头文件辅助。
[相关主题]: 模板元编程, 编译模型与预处理, C++20 协程, 构建系统（CMake）
[标签]: C++20, 模块, 编译模型, 工程实践, 性能优化

metedata: {'chunk_type': 'principle', 'role': 'cpp', 'source': 'cpp_chunks.jsonl', 'topic': 'C++20'}



In [23]:
query = "C++ 右值引用和移动语义"
docs_found = db.similarity_search(
    query = query, 
    k=3,
)
for doc in docs_found:
    print(f"id:{doc.id}\npage_content:{doc.page_content}\nmetedata:{doc.metadata}")

id:cpp_chunks_203
page_content:[知识点]: std::forward
[知识分类]: cpp
[内容]: std::forward 是 C++11 引入的完美转发工具，其核心原理是基于引用折叠规则和模板参数推导，将函数参数的值类别（左值或右值）原样传递给其他函数，避免不必要的拷贝或移动。概念上，它用于泛型编程中，确保在模板函数内部转发参数时，保持其原始的引用类型（如左值引用或右值引用）。关键机制：当模板参数 T 被推导为左值引用（如 T&）时，std::forward<T>(arg) 返回左值引用；当 T 被推导为非引用或右值引用（如 T 或 T&&）时，返回右值引用，从而支持移动语义。常见误区包括：误用 std::forward 与 std::move 混淆，std::move 无条件转换为右值，而 std::forward 有条件保留值类别；在非模板函数中使用 std::forward 无意义，因为它依赖模板参数推导。工程实践中，std::forward 是实现完美转发模式（如工厂函数或包装器）的基础，能显著提升性能，但需注意模板参数推导的细节，避免在复杂继承场景中失效。
[关键点]: std::forward 基于引用折叠规则，保持参数的原始值类别（左值或右值）。, 模板参数 T 的推导决定转发结果：T& 为左值，T 或 T&& 为右值。, 与 std::move 不同，std::forward 是条件性的，避免无谓的移动语义。, 在模板函数中使用，确保泛型代码的完美转发，减少拷贝开销。, 常见错误：在非模板上下文使用，或忽略模板参数推导导致转发失败。
[相关主题]: std::move, 引用折叠规则, 完美转发模式, 移动语义
[标签]: C++11, 模板元编程, 内存模型, 值类别, 泛型编程

metedata:{'chunk_type': 'principle', 'source': 'cpp_chunks.jsonl', 'topic': 'std::forward', 'role': 'cpp'}
id:cpp_chunks_4
page_content:[知识点]: C++ Standards
[知识分类]: cpp
[内容]: C++ 标准（如 C++11/14/17/20/23）在工程实践中核心价值在于提升

In [ ]:
query = "C++ 右值引用和移动语义"
docs_found = db.similarity_search(
    query = query, 
    k=3,
    filter={"topic":"C++11"},
)
"""
    filter={"tags":{"$contains": "C++11"}}
    这是处理同一metedata中有多个字段，但是我们只检测一个的情况
    方法是 "metadata":{"$contains":"label"}

    filter={
        "$and": [
            {"topic": ""},
            {"tags": ""}
        ]
"""
for doc in docs_found:
    print(f"id:{doc.id}\npage_content:{doc.page_content}\nmetedata:{doc.metadata}")

id:cpp_chunks_6
page_content:[知识点]: C++11
[知识分类]: cpp
[内容]: C++11 引入的右值引用（Rvalue Reference）和移动语义（Move Semantics）是工程实践中优化资源管理的关键特性。核心概念是区分左值（有名字的持久对象）和右值（临时或即将销毁的对象），通过 `T&&` 捕获右值，实现资源的高效转移而非复制。常见应用包括自定义容器、智能指针和工厂模式中，避免不必要的深拷贝，提升性能。工程实践中，移动构造函数和移动赋值运算符通常标记为 `noexcept` 以支持标准库容器的异常安全操作。常见误区包括：1）误将左值绑定到右值引用，导致编译错误；2）忘记实现移动语义，导致类对象仍使用拷贝语义，性能低下；3）在移动后未将源对象置为有效但未指定状态，引发未定义行为；4）过度使用移动，忽略拷贝构造在某些场景下更合适（如小型对象）。面试追问角度：如何为自定义资源管理类（如文件句柄）实现移动语义？移动语义与完美转发（Perfect Forwarding）如何结合？在模板中如何避免引用折叠导致的意外行为？
[关键点]: 右值引用用于捕获临时对象，避免不必要的拷贝, 移动构造函数应转移资源所有权并置源对象为有效状态, 标记 `noexcept` 以支持标准库容器的异常安全移动, 误区：左值绑定右值引用或忽略移动语义导致性能问题, 结合完美转发在模板中高效传递参数
[相关主题]: 完美转发, 智能指针, 模板元编程, 异常安全
[标签]: 右值引用, 移动语义, C++11优化, 资源管理, 工程实践

metedata:{'topic': 'C++11', 'role': 'cpp', 'chunk_type': 'practice', 'source': 'cpp_chunks.jsonl'}
id:cpp_chunks_5
page_content:[知识点]: C++11
[知识分类]: cpp
[内容]: C++11 引入的移动语义（Move Semantics）是解决资源管理效率问题的核心机制，其原理基于右值引用（rvalue reference）和移动构造函数/移动赋值运算符。概念上，移动语义允许将资源（如内存、文件句柄）的所有权从临时对象（右值）高效转移，而非复制，从而避免不必要的深拷贝开销。关

In [50]:
query = "C++11 引入的右值引用（Rvalue Reference）和移动语义（Move Semantics）"
docs_found = db.similarity_search_with_score(
    query = query, 
    k=3,
    # filter={"topic":"C++11"},
)
print(type(docs_found))
for doc in docs_found:
    Doc , score = doc[0] , doc[1]
    print(f"id:{Doc.id}\npage_content:{Doc.page_content}\nmetedata:{Doc.metadata}")
    print(f"score={score}")

<class 'list'>
id:cpp_chunks_196
page_content:[知识点]: Move Semantics
[知识分类]: cpp
[内容]: 移动语义（Move Semantics）是C++11引入的核心特性，旨在通过转移资源所有权而非复制来提升性能，尤其适用于临时对象或大对象场景。在工程实践中，常见应用包括实现高效容器操作（如vector扩容时移动元素）、自定义资源管理类（如智能指针、文件句柄）以及优化函数返回值（避免不必要的拷贝）。关键原理是利用右值引用（&&）和std::move将对象标记为“可移动”，从而调用移动构造函数或移动赋值运算符，转移内部资源（如指针、缓冲区）而非深拷贝。常见误区包括：误用std::move导致悬空引用（如移动后访问原对象）、在const对象上使用移动（应使用拷贝）、以及未正确实现移动操作（如遗漏noexcept修饰导致异常不安全）。工程实践建议：为资源管理类遵循Rule of Five（定义析构、拷贝、移动构造/赋值），优先使用移动语义优化性能，并通过静态分析工具检测误用。面试追问角度可聚焦于移动语义与异常安全、与完美转发的结合、以及在多线程环境下的资源转移风险。
[关键点]: 移动语义通过右值引用转移资源所有权，避免深拷贝开销。, 工程中常用于容器操作、自定义资源类和函数返回值优化。, 常见误区：误用std::move导致悬空引用或在const对象上移动。, 实践建议：遵循Rule of Five，确保移动操作noexcept以保证异常安全。, 面试可追问移动语义与完美转发、多线程资源管理的关联。
[相关主题]: 右值引用, 完美转发, Rule of Five, 异常安全
[标签]: C++11, 移动语义, 性能优化, 资源管理, 面试考点

metedata:{'source': 'cpp_chunks.jsonl', 'role': 'cpp', 'chunk_type': 'practice', 'topic': 'Move Semantics'}
score=0.6797518730163574
id:cpp_chunks_5
page_content:[知识点]: C++11
[知识分类]: cpp
[内容]: C++11 引入的移动语义（Move Semantics）是解决资源管理效率

In [ ]:
query = "C++"
docs_found = db.similarity_search_with_score(
    query = query, 
    k=3,
    # filter={"topic":"C++11"},
)
print(type(docs_found))
for doc in docs_found:
    Doc , score = doc[0] , doc[1]
    # if score < 1.0 可以做过滤
    print(f"id:{Doc.id}\npage_content:{Doc.page_content}\nmetedata:{Doc.metadata}")
    print(f"score={score}")
""" 
对比可以发现，这里的score是距离而非相似度
更精确的query的score更高 更长
"""

<class 'list'>
id:cpp_chunks_206
page_content:[知识点]: Range-based For
[知识分类]: cpp
[内容]: Range-based for 循环是 C++11 引入的语法糖，用于遍历容器或范围，其核心是编译器将其展开为迭代器循环。在工程实践中，它简化了代码，但需注意底层机制以避免陷阱。常见应用包括遍历 `std::vector`、`std::map` 等容器，例如 `for (auto& elem : container) { /* 使用 elem */ }`。关键原理是：循环语句 `for (auto&& elem : range)` 会被编译为 `for (auto it = std::begin(range); it != std::end(range); ++it) { auto&& elem = *it; /* 循环体 */ }`，这依赖于 `std::begin` 和 `std::end` 的 ADL（参数依赖查找）。工程实践中，必须使用引用（`auto&` 或 `auto&&`）避免不必要的拷贝，尤其对大型对象；对于 `const` 容器，使用 `const auto&`。常见误区包括：1) 遍历 `std::vector<bool>` 时，由于代理对象问题，`auto&` 可能导致编译错误，应使用 `auto` 或显式转换；2) 在循环中修改容器（如 `erase`）会导致迭代器失效，应使用 `erase-remove` 模式或 C++20 的 `std::erase_if`；3) 误用 `auto` 而非引用，导致性能下降或意外修改；4) 遍历关联容器（如 `std::map`）时，`auto` 默认是 `std::pair<const Key, Value>`，需注意 `const` 键不可修改。面试追问角度：如何自定义类型支持 range-based for？需实现 `begin()` 和 `end()` 成员函数或提供非成员版本；在多线程环境下，遍历共享容器时如何保证安全？需结合锁或原子操作。
[关键点]: Range-based for 是语法糖，编译为迭代器循环，依赖 `std::begin` 和 `std::end`。, 工程中必须使用引用（`auto&` 或 `auto

In [57]:
"""
topic更分散 
lambda_mult 0.1 更重视多样性    topic无相同
lambda_mult 0.9 更接近普通检索  实际使用中可以发现topic有相同
"""
docs_mmr = db.max_marginal_relevance_search(
    query,
    k=5,
    fetch_k=20,
    lambda_mult=0.1
)
for doc in docs_mmr:
    print(f"page_content:{doc.page_content}..., metadata={doc.metadata}, id={doc.id}")

page_content:[知识点]: Move Semantics
[知识分类]: cpp
[内容]: 移动语义（Move Semantics）是C++11引入的核心特性，旨在通过转移资源所有权而非复制来提升性能，尤其适用于临时对象或大对象场景。在工程实践中，常见应用包括实现高效容器操作（如vector扩容时移动元素）、自定义资源管理类（如智能指针、文件句柄）以及优化函数返回值（避免不必要的拷贝）。关键原理是利用右值引用（&&）和std::move将对象标记为“可移动”，从而调用移动构造函数或移动赋值运算符，转移内部资源（如指针、缓冲区）而非深拷贝。常见误区包括：误用std::move导致悬空引用（如移动后访问原对象）、在const对象上使用移动（应使用拷贝）、以及未正确实现移动操作（如遗漏noexcept修饰导致异常不安全）。工程实践建议：为资源管理类遵循Rule of Five（定义析构、拷贝、移动构造/赋值），优先使用移动语义优化性能，并通过静态分析工具检测误用。面试追问角度可聚焦于移动语义与异常安全、与完美转发的结合、以及在多线程环境下的资源转移风险。
[关键点]: 移动语义通过右值引用转移资源所有权，避免深拷贝开销。, 工程中常用于容器操作、自定义资源类和函数返回值优化。, 常见误区：误用std::move导致悬空引用或在const对象上移动。, 实践建议：遵循Rule of Five，确保移动操作noexcept以保证异常安全。, 面试可追问移动语义与完美转发、多线程资源管理的关联。
[相关主题]: 右值引用, 完美转发, Rule of Five, 异常安全
[标签]: C++11, 移动语义, 性能优化, 资源管理, 面试考点
..., metadata={'chunk_type': 'practice', 'topic': 'Move Semantics', 'source': 'cpp_chunks.jsonl', 'role': 'cpp'}, id=cpp_chunks_196
page_content:[知识点]: Design Patterns in C++
[知识分类]: cpp
[内容]: 在C++中，设计模式是解决常见软件设计问题的可复用方案，其核心原理在于通过封装变化、解耦依赖和提升扩展性来应对复杂性。以单例模式为例，其原理是确

In [71]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
docs = retriever.invoke("C++ 智能指针和 RAII")
print(docs)
for doc in docs:
    print(f"ids:{doc.id}\npage_content:{doc.page_content}\n metadata={doc.metadata}\n")

[Document(id='llm_core_tech_chunks_69', metadata={'role': 'llm_core_tech', 'chunk_type': 'principle', 'topic': 'Multi-Agent Systems', 'source': 'llm_core_tech_chunks.jsonl'}, page_content='[知识点]: Multi-Agent Systems\n[知识分类]: llm_core_tech\n[内容]: Multi-Agent Systems（多智能体系统）是指由多个自主或半自主的智能体（Agent）组成的分布式计算系统，这些智能体通过交互、协作或竞争来完成复杂任务。其核心原理在于：单个智能体的能力有限，通过多智能体协作可以实现单智能体无法完成的任务，例如分布式决策、资源分配和复杂问题求解。关键机制包括通信协议（如消息传递、共享黑板）、协调策略（如合同网协议、拍卖机制）和学习机制（如多智能体强化学习）。在大语言模型（LLM）背景下，多智能体系统常用于构建对话系统、模拟社会行为或解决复杂推理任务，其中每个智能体可能是一个LLM实例，通过提示工程和角色分配实现分工协作。\n\n常见误区包括：认为多智能体系统只是简单地将多个LLM并行运行，而忽略了智能体间的交互协议设计；或过度依赖集中式控制，导致系统失去分布式优势。工程实践中，需平衡通信开销与协作效率，避免智能体间死锁或目标冲突。原理上，多智能体系统体现了分布式人工智能的核心思想，即通过局部自治和全局协调实现涌现智能。\n[关键点]: 多智能体系统由多个自主智能体组成，通过交互完成复杂任务。, 核心机制包括通信协议、协调策略和学习机制。, 在LLM中，多智能体通过角色分配和提示工程实现分工协作。, 常见误区是忽略交互协议设计或过度集中控制。, 工程实践需平衡通信开销与协作效率，避免死锁。\n[相关主题]: 强化学习, 分布式系统, 提示工程, 对话系统\n[标签]: 多智能体, LLM, 分布式AI, 协作机制, 强化学习\n'), Document(id='cpp_chunks_112', metadata={'topic': 'Use-After-Free', 'role': 'cpp', 'source': 'cpp_chunks.

In [72]:
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":4,
        "fetch_k":10,
        "lambda_mult":0.1,
    }
)

docs = retriever.invoke(query)
for doc in docs:
    print(doc.page_content, doc.metadata, doc.id)

[知识点]: Move Semantics
[知识分类]: cpp
[内容]: 移动语义（Move Semantics）是C++11引入的核心特性，旨在通过转移资源所有权而非复制来提升性能，尤其适用于临时对象或大对象场景。在工程实践中，常见应用包括实现高效容器操作（如vector扩容时移动元素）、自定义资源管理类（如智能指针、文件句柄）以及优化函数返回值（避免不必要的拷贝）。关键原理是利用右值引用（&&）和std::move将对象标记为“可移动”，从而调用移动构造函数或移动赋值运算符，转移内部资源（如指针、缓冲区）而非深拷贝。常见误区包括：误用std::move导致悬空引用（如移动后访问原对象）、在const对象上使用移动（应使用拷贝）、以及未正确实现移动操作（如遗漏noexcept修饰导致异常不安全）。工程实践建议：为资源管理类遵循Rule of Five（定义析构、拷贝、移动构造/赋值），优先使用移动语义优化性能，并通过静态分析工具检测误用。面试追问角度可聚焦于移动语义与异常安全、与完美转发的结合、以及在多线程环境下的资源转移风险。
[关键点]: 移动语义通过右值引用转移资源所有权，避免深拷贝开销。, 工程中常用于容器操作、自定义资源类和函数返回值优化。, 常见误区：误用std::move导致悬空引用或在const对象上移动。, 实践建议：遵循Rule of Five，确保移动操作noexcept以保证异常安全。, 面试可追问移动语义与完美转发、多线程资源管理的关联。
[相关主题]: 右值引用, 完美转发, Rule of Five, 异常安全
[标签]: C++11, 移动语义, 性能优化, 资源管理, 面试考点
 {'chunk_type': 'practice', 'source': 'cpp_chunks.jsonl', 'topic': 'Move Semantics', 'role': 'cpp'} cpp_chunks_196
[知识点]: Rvalue References
[知识分类]: cpp
[内容]: Rvalue References（右值引用）是 C++11 引入的核心机制，用于支持移动语义和完美转发。其核心原理是通过类型系统区分左值（lvalue）和右值（rvalue），右值引用（T&&）专门绑定到临时对象或即将销毁的对象，从而避免

# RAG效果评估

## 指标
### 1。Recall 
前k个检索结果中，是否包含正确的文档

### 2.Precision@k
前k个结果中有多少是真的相关的

### 3.